# Movie-Grounded Creative Director — Real-Qwen Validation

Validates `MovieGroundedDirector` against the **existing** Movie Intelligence
for project **`bc6384be-47a5-4ee8-8674-7ff861472026`** using real Qwen on a T4.

What this notebook records (honestly, only what is measured):

- GPU / VRAM / device / model / dtype
- model load time + per-call generation times (from the Qwen provider)
- wall-clock, number of concepts, rejected, regenerated, selected concept

**Before starting**: upload the project folder
`data/bc6384be-47a5-4ee8-8674-7ff861472026` (it contains `movie_index.json`)
into Google Drive at `MyDrive/automovies_movie_intel/bc6384be-47a5-4ee8-8674-7ff861472026`.

Runtime: **Runtime → Change runtime type → T4 GPU**.

In [ ]:
# 1) GPU check — must show a T4
!nvidia-smi

In [ ]:
# 2) Repo + Qwen dependencies (idempotent)
%cd /content
if not __import__('os').path.exists('/content/automovies'):
    !git clone --depth 1 https://github.com/asdfhgds/automovies.git automovies
%cd /content/automovies
!git pull --ff-only 2>/dev/null || true
!python -m pip install -q "transformers>=4.52,<5" accelerate sentencepiece protobuf bitsandbytes
import torch
assert torch.cuda.is_available(), 'No GPU visible - set Runtime type to T4'
print('torch', torch.__version__, '| gpu', torch.cuda.get_device_name(0))

In [ ]:
# 3) Mount Drive and point at the uploaded bc6384be project
from google.colab import drive
drive.mount('/content/drive')

PROJECT_DIR = '/content/drive/MyDrive/automovies_movie_intel/bc6384be-47a5-4ee8-8674-7ff861472026'
import os
assert os.path.exists(PROJECT_DIR + '/movie_index.json'), (
    'movie_index.json not found at ' + PROJECT_DIR + ' - upload the project folder first')
print('movie intelligence found at', PROJECT_DIR)

In [ ]:
# 4) Run the real-Qwen director validation (strict: real LLM required)
import os

os.environ['REQUIRE_REAL_LLM'] = 'true'            # refuses mock/deterministic fallback
os.environ['DIRECTOR_MODEL'] = 'Qwen/Qwen3-4B-Instruct-2507'
os.environ['DIRECTOR_DTYPE'] = '4bit'              # NF4 to be safe on a 16GB T4
os.environ['DIRECTOR_TEMPERATURE'] = '0.8'

!python scripts/run_director_validation.py --project "$PROJECT_DIR" --num-concepts 5 --min-coverage 0.4 --duration-sec 90

In [ ]:
# 5) Show the outputs written by the harness
import json, os
p = '/content/drive/MyDrive/automovies_movie_intel/bc6384be-47a5-4ee8-8674-7ff861472026'
for f in ('reports/director_reasoning.md', 'reports/director_validation.json'):
    print('='*30, f)
    print(open(p + '/' + f, encoding='utf-8').read())
v = json.load(open(p + '/reports/director_validation.json', encoding='utf-8'))
print('='*30, 'runtime')
print(json.dumps(v.get('runtime', {}), indent=2, ensure_ascii=False))

## What to copy back to the repo

1. `reports/director_reasoning.md` and `reports/director_validation.json` from the
   project folder (or paste their contents into the local repo).
2. Fill `reports/director_validation.md` / `.json` (repo root) human-eval fields
   per concept: `specificity`, `grounding`, `originality`, `visual_potential`,
   `generic_ai_feeling`, `human_notes` (GOOD/PARTIAL/BAD and LOW/MEDIUM/HIGH).

## Interpretation cheatsheet

- **PASS**: 5 genuinely different, specific, grounded, visually useful concepts.
- **PARTIAL**: grounded but still generic, or evidence matching is weak
  (e.g. `revolver` vs `weapon` synonym misses).
- **FAIL**: mostly hallucinated / generic / unrelated to the movie.

Also inspect whether rejected concepts are the *right* rejects (un-evidenced,
generic) and whether the selected concept's `evidence_strategy.scene_ids` map to
real scenes with real relevance.